##ML Guitar Effect Classification Project
CMPU-395: Machine Learning

Axel Estrada




In [ ]:
# @title Import (RUN THIS FIRST)

import torch
import torchaudio

from torch import nn
from torch.cuda import is_available
from torch.utils.data import DataLoader
from torch.utils.data import Dataset

from torchvision.transforms import ToTensor
from torchvision import datasets
from torchsummary import summary

import pandas as pd
from pathlib import Path # for Colab

In [ ]:
# @title Global Constants

# TRAINING
BATCH_SIZE = 128
EPOCHS = 10
LEARNING_RATE = 0.001

# DATA LOADER
SAMPLE_RATE = 22050
NUM_SAMPLES = 22050

# FILE LOCATIONS
MEL_SPECTROGRAMS_FILES_DIR =
AUDIO_DIR = None

EFFECT_MAP = {
    "808": "overdrive",
    "TS9": "overdrive",
    "BD2": "overdrive",
    "OD1": "overdrive",
    "SD1": "overdrive",
    "MGS": "overdrive",

    "DS1": "distortion",
    "RAT": "distortion",
    "DPL": "distortion",
    "MT2": "distortion",

    "FFC": "fuzz",
    "BMF": "fuzz",
    "RBM": "fuzz",
    "VTB": "fuzz"
}

In [ ]:
# @title Loading Mel-spectograms
from google.colab import drive



In [ ]:
# @title Data Loader Class
# Note: Much of this is for generating the Mel-spectograms
# and cleaning up .wav files if necessary
# Most of the spectograms are provided by the data set authors,
# and are instead uploaded through google drive
class GUITAR_X_DIST_DataSet(Dataset):

    def __init__(self,
                 annotations_file,
                 audio_dir,
                 transformation,
                 target_sample_rate,
                 num_samples,
                 device):
        self.annotations = pd.read_csv(annotations_file)
        self.audio_dir = audio_dir
        self.device = device
        self.transformation = transformation.to(self.device) # GPU or CPU device
        self.target_sample_rate = target_sample_rate
        self.num_samples = num_samples

    # how to use the len syntax: len(usd)
    def __len__(self):
        return len(self.annotations)

    # a_list[1] -> a_list.__getitem__(1)
    def __getitem__(self, index):
        audio_sample_path = self._get_audio_sample_path(index)
        label = self._get_audio_sample_label(index)

        # signal is the waveform time series; sr is sample rate
        signal, sr = torchaudio.load(audio_sample_path)

        # assigning signal to be passed to device
        signal = signal.to(self.device)

        # need to resample all data we have to ensure everything has same sample rate
        # otherwise melspectograms will be inconsistent in their dimensions
        signal = self._resample_if_necessary(signal, sr)

        # not all data will be mono, could be multi-channel audio
        signal = self._mix_down_if_necessary(signal)

        # want to ensure signal has correct n_samples
        signal = self._cut_if_necessary(signal)         # if too long
        signal = self._right_pad_it_necessary(signal)   # if too short

        signal = self.transformation(signal) # equivalent to mel_spectrogram(signal)

        return signal, label

    def _get_audio_sample_path(self, index):
        fold = f"fold{self.annotations.iloc[index, 5]}"

        # LOCAL METHOD
        #path = os.path.join(self.audio_dir, fold, self.annotations.iloc[
        #    index, 0])

        # COLAB METHOD
        path = Path(self.audio_dir) / fold / self.annotations.iloc[index, 0]

        return path

    def _get_audio_sample_label(self, index):
        return self.annotations.iloc[index, 6] # 6 is based on .csv column location

    def _resample_if_necessary(self, signal, sr):
        if sr != self.target_sample_rate:
            # .to(device) needed here to ensure it's same as signal device
            resampler = torchaudio.transforms.Resample(sr, self.target_sample_rate).to(device)
            signal = resampler(signal)

        return signal

    def _mix_down_if_necessary(self, signal):
        if signal.shape[0] > 1:
            signal = torch.mean(signal, dim=0, keepdim=True)
            # signal => (num_channels, samples) -> (2, 16000) -> (1, 16000)

        return signal

    def _cut_if_necessary(self, signal):
        # signal -> Tensor -> (1, num_samples) -> (1, 50000) -> (1, 22050)
        if signal.shape[1] > self.num_samples:
            # keep 1st dim untouched
            signal = signal[:, :self.num_samples]

        return signal

    def _right_pad_it_necessary(self, signal):
        length_signal= signal.shape[1]
        if length_signal < self.num_samples:
            num_missing_samples = self.num_samples - length_signal

             # [1, 1, 1] -> [0, 1, 1, 1, 0, 0]
            last_dim_padding = (0, num_missing_samples) # (1, 2)
            signal = torch.nn.functional.pad(signal, last_dim_padding)

        return signal

In [ ]:
# @title CNN Model (VGG)
# need to look into optimizing performance for this

class CNNNetwork(nn.Module):

    def __init__(self):
        super().__init__()
        # 4 conv blocks / flatten / linear / softmax
        self.conv1 = nn.Sequential(
            nn.Conv2d(
                in_channels=1,
                out_channels=16,
                kernel_size=3,
                stride=1,
                padding=2
            ),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2)
        )

        self.conv2 = nn.Sequential(
            nn.Conv2d(
                in_channels=16,
                out_channels=32,
                kernel_size=3,
                stride=1,
                padding=2
            ),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2)
        )

        self.conv3 = nn.Sequential(
            nn.Conv2d(
                in_channels=32,
                out_channels=64,
                kernel_size=3,
                stride=1,
                padding=2
            ),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2)
        )

        self.conv4 = nn.Sequential(
            nn.Conv2d(
                in_channels=64,
                out_channels=128,
                kernel_size=3,
                stride=1,
                padding=2
            ),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2)
        )

        self.flatten = nn.Flatten()
        self.linear = nn.Linear(128 * 5 * 4, 10)
        self.softmax = nn.Softmax(dim=1)

    def forward(self, input_data):
        x = self.conv1(input_data)
        x = self.conv2(x)
        x = self.conv3(x)
        x = self.conv4(x)
        x = self.flatten(x)
        logits = self.linear(x)

        # Don't need if we're using Cross_Entropy_Loss()
        # predictions = self.softmax(logits)
        return logits

In [ ]:
# @title Model Training Functions

def train_one_epoch(model, data_loader, loss_fn, optimizer, device):
    for inputs, targets in data_loader:
        # have to assign tensors to device we're using
        inputs, targets = inputs.to(device), targets.to(device)

        # calculate loss
        predictions = model(inputs)
        loss = loss_fn(predictions, targets)

        # backpropogate loss and update weights
        optimizer.zero_grad() # reset gradients after each batch
        loss.backward()       # applies back propogation
        optimizer.step()      # updates weights

    # print loss for last batch
    print(f"loss: {loss.item()}")

def train(model, data_loader, loss_fn, optimizer, device, n_epochs):
    for i in range(n_epochs):
        print(f"Epoch {i+1}")
        train_one_epoch(model, data_loader, loss_fn, optimizer, device)
        print("------------")

    print("Training is done!")

In [ ]:
# @title Data Loader

def create_data_loader(train_data, batch_size):
    train_data_loader = DataLoader(train_data, batch_size=BATCH_SIZE)
    return train_data_loader
# Note: you'd also do this for the validation data

# Note 2: Need to update this part to work with the .npy files we already have

In [ ]:
# @title Important Setup + Creating & Training Model

# setting up model device
if torch.cuda.is_available():
    device = "cuda"
else:
    device = "cpu"
print(f"Using {device} device")

# creating model
cnn = CNNNetwork().to(device)

# instantiate loss function + optimizer
loss_fn = nn.CrossEntropyLoss() # remember that this already uses softmax
optimizer = torch.optim.Adam(cnn.parameters(),
                             lr=LEARNING_RATE)

# train the model
train(cnn, train_data_loader, loss_fn, optimizer, device, EPOCHS)

In [ ]:
# @title Saving Our Model

# state_dict() is a python dictonary holding all layer parameters
torch.save(cnn.state_dict(), "cnnnet.pth")
print("Model trained and stored at cnnnet.pth")

In [ ]:
# @title Testing Our Model

# Note: Need to update this majorly so that it tests over an entire test set
def predict(model, input, target, class_mapping):
    # switch that changes how pytorch model behaves (gets rid of certain layers)
    # model.train() would be for going back to training it
    model.eval()

    # context manager that makes model not calculate any gradients
    # while performing the following commands
    with torch.no_grad():
        predictions = model(input)
        # Tensor (# of inputs, # of classes)
        # Tensor (1, 10) -> [ [0.1, 0.01, ..., 0.6] ]
        predicted_index = predictions[0].argmax() # index with highest value
        predicted = class_mapping[predicted_index]
        expected = class_mapping[target]

    return predicted, expected

cnn.to(device) # make sure device matches everywhere
predicted, expected = predict(cnn, input, target,
                              EFFECT_MAP) # class_mapping is mapping integers to classes

print(f"Predicted: '{predicted}', expected: '{expected}'")